In [2]:
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_validate
from xgboost import XGBRegressor

# -------------------------------------------------------------
# [필수 준비] 혹시 데이터나 변수가 유실되었을 경우를 대비한 정의
# (만약 이미 앞단에서 정의하셨다면 이 부분은 생략되거나 그대로 유지됩니다)
# -------------------------------------------------------------
# TARGET = '당신이 설정한_타겟컬럼명'
# UP = [상류 센서 및 원자재 속성 컬럼 리스트 전체]

# 1. 원자재 속성(로트 고정값)을 제외한 순수 공정 센서 목록(PROC) 정의
PROC = [c for c in UP if "Raw_Prop" not in c]


# 2. 특징셋 평가를 위한 함수 정의
def evaluate(F, name):
  # 타겟 데이터를 붙이고 결측치 제거
  D = F.assign(y=df[TARGET]).dropna()

  # 10분 단위 블록(600초)으로 그룹 설정 (데이터 누수 방지용)
  groups = D.index // 600

  # XGBoost 회귀 모델 정의
  m = XGBRegressor(
      n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0
  )

  # GroupKFold 교차 검증 수행
  r = cross_validate(
      m,
      D.drop(columns="y"),
      D["y"],
      cv=GroupKFold(5),
      groups=groups,
      scoring=("neg_root_mean_squared_error", "r2"),
  )

  return {
      "특징셋": name,
      "변수 수": F.shape[1],
      "RMSE": -r["test_neg_root_mean_squared_error"].mean(),
      "R2": r["test_r2"].mean(),
  }


# 3. 4가지 특징셋(Feature Sets) 데이터프레임 생성
F_all = df[UP]  # A. 상류 전체
F_proc = df[PROC]  # B. 원자재 속성 제외
F_lag = F_proc.assign(
    M2_MotorAmperage_U_lag15=df["M2_MotorAmperage_U"].shift(15),
    Comb_Temperature1_U_lag10=df["Comb_Temperature1_U"].shift(10),
)  # C. + 시차
F_ma = pd.concat(
    [F_proc, F_proc.rolling(60).mean().add_suffix("_ma60")], axis=1
)  # D. + 이동평균

# 4. 모델 평가 실행 및 결과 출력
fe_result = pd.DataFrame([
    evaluate(F_all, "A. 상류 전체"),
    evaluate(F_proc, "B. 원자재 속성 제외"),
    evaluate(F_lag, "C. B + 시차"),
    evaluate(F_ma, "D. B + 이동평균"),
])

print(fe_result.round(4))

NameError: name 'UP' is not defined